# Model 2: Fine-tuned Model (No API)

**Goal:** Fine-tune a German GPT-2 model on Austrian tax law Q&A examples, then run it on all 644 questions.

**Training data:** 152 Q&A pairs hand-written from the actual Austrian tax law texts (KStG, EStG, UStG). Stored in `training_data.csv` — no API used to generate them.

**3 steps:**
1. Load local training data from `training_data.csv`
2. Fine-tune `dbmdz/german-gpt2` using HuggingFace Trainer on Colab GPU
3. Run the fine-tuned model on all 644 questions and download results

In [ ]:
# Install required packages
!pip install transformers datasets pandas torch --quiet

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Check GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 1: Load Training Data

Training data is a local CSV with 152 Q&A pairs written from the Austrian tax law texts (KStG 1988, EStG 1988, UStG 1994). No API was used to generate them. Zero overlap with the test dataset (dataset_clean.csv).

In [ ]:
# Load training data from local CSV
# If running on Colab: upload training_data.csv first, or load from GitHub
# Option A (local/Colab upload):
train_df = pd.read_csv("training_data.csv")

# Option B (from GitHub — uncomment if running on Colab without upload):
# train_df = pd.read_csv("https://raw.githubusercontent.com/frederrrrr/wu-llms-ss26/main/VAT-INTL-001_Samorokov_Puthenparambil_Ertl/h12406664_Fedor_Samorokov/code/training_data.csv")

print(f"Loaded {len(train_df)} training Q&A pairs")
print(f"\nExample:")
print(f"Q: {train_df.iloc[0]['question']}")
print(f"A: {train_df.iloc[0]['answer']}")

In [ ]:
# Format training data as text: "Frage: X\nAntwort: Y"
# This is the standard format for causal language model fine-tuning
texts = [f"Frage: {row['question']}\nAntwort: {row['answer']}" for _, row in train_df.iterrows()]

train_dataset = Dataset.from_dict({"text": texts})
print(f"Training examples: {len(train_dataset)}")
print(f"\nExample:\n{texts[0]}")

## Step 2: Fine-tune the Model

We fine-tune `dbmdz/german-gpt2` — a GPT-2 model pre-trained on German text.
All training happens locally on the GPU — no API calls.

In [ ]:
# Load tokenizer and model
model_name = "dbmdz/german-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.resize_token_embeddings(len(tokenizer))
print(f"Model loaded: {model_name}")

In [ ]:
# Tokenize the training data
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(f"Tokenized {len(tokenized_dataset)} examples")

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./finetuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy="no",
    report_to="none"    
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Train the model
trainer.train()

In [ ]:
# Save the fine-tuned model
trainer.save_model("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")
print("Model saved to ./finetuned_model")

## Step 3: Run Inference on 644 Questions

We load the dataset directly from GitHub and generate answers using our fine-tuned model.

In [ ]:
# Load dataset from GitHub
df = pd.read_csv("https://raw.githubusercontent.com/frederrrrr/wu-llms-ss26/main/VAT-INTL-001_Samorokov_Puthenparambil_Ertl/h12406664_Fedor_Samorokov/dataset_clean.csv")
print(f"Loaded {len(df)} questions")
df.head(3)

In [ ]:
# Load fine-tuned model for inference
ft_tokenizer = AutoTokenizer.from_pretrained("./finetuned_model")
ft_model = AutoModelForCausalLM.from_pretrained("./finetuned_model").to(device)
ft_model.eval()
print("Fine-tuned model loaded")

In [ ]:
# Generate answers for all 644 questions
answers = []

for i, row in df.iterrows():
    prompt = f"Frage: {row['prompt']}\nAntwort:"
    inputs = ft_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=200).to(device)

    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs,
            max_new_tokens=150,
            pad_token_id=ft_tokenizer.eos_token_id,
            do_sample=False
        )

    full_text = ft_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Extract only the answer part (everything after "Antwort:")
    answer = full_text.split("Antwort:")[-1].strip()
    answers.append(answer)
    print(f"{i+1}/{len(df)} | {row['id']} | {answer[:70]}...")

In [ ]:
# Save results CSV
results = pd.DataFrame({"id": df["id"], "answer": answers})
results.to_csv("model2_results.csv", index=False)
print(f"Saved {len(results)} answers")
results.head(3)

In [ ]:
# Download the results CSV
from google.colab import files
files.download("model2_results.csv")
print("Download started — move the file to VAT-INTL-001/results/")